# Lecția 10 - Agenți AI în producție

În această lecție veți învăța **modele de producție** pentru agenții AI utilizând Microsoft Agent Framework (Python). Acoperim:

- **Observabilitatea** — adăugarea de temporizări și înregistrări la interacțiunile agentului
- **Evaluarea** — utilizarea unui agent evaluator pentru a puncta calitatea răspunsurilor
- **Gestionarea costurilor** — strategii pentru optimizarea tokenilor și selecția modelului

Scenariul este un **agent de turism** care ajută utilizatorii să planifice călătorii, cu monitorizare și evaluare suprapuse.


## Configurare


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import time
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Considerații privind producția

Mutarea agenților AI de la prototipuri la producție necesită o atenție atentă asupra a trei piloni:

1. **Observabilitate** — Aveți nevoie de vizibilitate asupra a ceea ce face agentul, cât durează și ce instrumente apelează. Fără trasare și jurnalizare, depanarea problemelor de producție este aproape imposibilă.

2. **Evaluare** — Verificările automate de calitate asigură că răspunsurile agentului rămân precise, complete și utile în timp. Un agent evaluator poate nota răspunsurile în funcție de criterii definite.

3. **Gestionarea costurilor** — Utilizarea tokenilor impactează direct costul. Strategii precum optimizarea cererii, selecția modelului și stocarea în cache ajută la menținerea cheltuielilor sub control fără a sacrifica calitatea.


## Construirea unui agent observabil

Definim unelte de călătorie și împachetăm apelul agentului cu cronometrare astfel încât să putem monitoriza latența. În producție, ați integra cu OpenTelemetry sau un backend de urmărire similar.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = client.as_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Modele de evaluare

Un model comun în producție este utilizarea unui al doilea agent ca **evaluator**. Evaluatorul acordă un scor răspunsului agentului principal în funcție de criterii prestabilite, cum ar fi completitudinea, acuratețea și utilitatea.

Acest lucru permite:
- Porți automate de calitate înainte ca răspunsurile să ajungă la utilizatori
- Detectarea regresiilor atunci când se modifică solicitările sau modelele
- Monitorizarea continuă a performanței agentului de-a lungul timpului


In [ ]:
evaluator = client.as_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategii de gestionare a costurilor

Controlul costurilor este esențial pentru agenții AI de producție. Iată strategiile cheie:

| Strategie | Descriere |
|---|---|
| **Optimizarea promptului** | Păstrați instrucțiunile sistemului concise. Eliminați contextul redundant pentru a reduce tokenii de intrare. |
| **Selecția modelului** | Folosiți modele mai mici, mai ieftine (de ex. GPT-4o-mini) pentru sarcini simple precum clasificarea sau extragerea, și rezervați modelele mai mari pentru raționamente complexe. |
| **Caching** | Cache-uiți rezultatele instrumentelor și interogările frecvente pentru a evita apelurile API redundante. |
| **Bugete pentru tokeni** | Stabiliți limite `max_tokens` pentru a preveni răspunsurile neașteptat de lungi. |
| **Agruparea cererilor (Batching)** | Grupați mai multe interogări ale utilizatorului într-un singur apel API, acolo unde este posibil. |

În practică, o abordare pe niveluri funcționează bine: direcționați cererile simple către un model rapid și ieftin și escaladați doar interogările complexe către unul mai capabil.


## Summary

În această lecție ai învățat cum să:

1. **Adaugi observabilitate** interacțiunilor agenților cu măsurarea timpului și înregistrarea, punând bazele pentru trasare și monitorizare.
2. **Evaluezi automat răspunsurile agenților** folosind un agent evaluator care punctează completitudinea, acuratețea și utilitatea.
3. **Gestionezi costurile** prin optimizarea solicitărilor, selecția modelului, stocarea în cache și bugetele de tokeni.

Aceste modele de producție ajută la asigurarea faptului că agenții tăi AI sunt de încredere, măsurabili și rentabili la scară largă.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
